In [1]:
import pandas as pd
import numpy as np
import re
import joblib

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file_path = "/content/projectAI.xlsx"

df = pd.read_excel(file_path, sheet_name="Sheet1", engine="openpyxl")

print("Số dòng, số cột:", df.shape)
df.head()

print("Danh sách cột:")
print(df.columns.tolist())

print("\nThông tin dữ liệu:")
df.info()

print("\nSố lượng dữ liệu thiếu:")
print(df.isnull().sum())

df = df.dropna(how="all")

# Kiểm tra lại số dòng
print("Số dòng sau khi xóa dòng trống:", len(df))

# Ép các cột số cần thiết về dạng numeric
numeric_cols = [
    "gan_truong",
    "gan_van_phong",
    "mat_tien",
    "trong_hem",
    "delivery",
    "wifi",
    "may_lanh",
    "cho_ngoi_lau",
    "dien_tich_num",
    "so_cho_ngoi_num",
    "so_nhan_vien_num",
    "rating_num",
    "so_review_num",
    "doi_thu_num",
    "vi_tri_score",
    "gia_tb_ngay_thuong",
    "gia_tb_cuoi_tuan",
    "doanh_thu_ngay_thuong",
    "doanh_thu_cuoi_tuan",
    "doanh_thu_tuan"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Xóa các dòng bị thiếu dữ liệu quan trọng
df = df.dropna(subset=numeric_cols)

print("Số dòng sau khi làm sạch:", len(df))
df.head()

features = [
    "loai_quan",
    "khu_vuc",
    "gan_truong",
    "gan_van_phong",
    "mat_tien",
    "trong_hem",
    "dien_tich_num",
    "so_cho_ngoi_num",
    "so_nhan_vien_num",
    "rating_num",
    "so_review_num",
    "delivery",
    "wifi",
    "may_lanh",
    "cho_ngoi_lau",
    "tep_khach",
    "gio_peak",
    "doi_thu_num",
    "vi_tri_score",
    "gia_tb_ngay_thuong",
    "gia_tb_cuoi_tuan"
]

target_ngay_thuong = "doanh_thu_ngay_thuong"
target_cuoi_tuan = "doanh_thu_cuoi_tuan"
target_tuan = "doanh_thu_tuan"

X = df[features]

y_ngay_thuong = df[target_ngay_thuong]
y_cuoi_tuan = df[target_cuoi_tuan]
y_tuan = df[target_tuan]

print("Kích thước X:", X.shape)
print("Kích thước y ngày thường:", y_ngay_thuong.shape)
print("Kích thước y cuối tuần:", y_cuoi_tuan.shape)
print("Kích thước y tuần:", y_tuan.shape)

X.head()

categorical_features = [
    "loai_quan",
    "khu_vuc",
    "tep_khach",
    "gio_peak"
]

numeric_features = [
    "gan_truong",
    "gan_van_phong",
    "mat_tien",
    "trong_hem",
    "dien_tich_num",
    "so_cho_ngoi_num",
    "so_nhan_vien_num",
    "rating_num",
    "so_review_num",
    "delivery",
    "wifi",
    "may_lanh",
    "cho_ngoi_lau",
    "doi_thu_num",
    "vi_tri_score",
    "gia_tb_ngay_thuong",
    "gia_tb_cuoi_tuan"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

print("Đã tạo preprocessor thành công!")

def train_regression_model(X, y, model_name):
    """
    Hàm này dùng để:
    - Chia train/test
    - Train Random Forest Regressor
    - Đánh giá model bằng MAE, RMSE, R2
    - Trả về model đã train
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                max_depth=None
            ))
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("==============================")
    print(f"KẾT QUẢ MODEL: {model_name}")
    print("==============================")
    print(f"MAE  : {mae:,.0f} VNĐ")
    print(f"RMSE : {rmse:,.0f} VNĐ")
    print(f"R2   : {r2:.4f}")

    return model, X_test, y_test, y_pred

model_ngay_thuong, X_test_ngay, y_test_ngay, y_pred_ngay = train_regression_model(
  X,
  y_ngay_thuong,
  "Dự đoán doanh thu ngày thường"
)
model_cuoi_tuan, X_test_cuoi_tuan, y_test_cuoi_tuan, y_pred_cuoi_tuan = train_regression_model(
    X,
    y_cuoi_tuan,
    "Dự đoán doanh thu cuối tuần"
)
model_tuan, X_test_tuan, y_test_tuan, y_pred_tuan = train_regression_model(
    X,
    y_tuan,
    "Dự đoán doanh thu tuần"
)
result_compare = pd.DataFrame({
    "Thuc_te_ngay_thuong": y_test_ngay.values,
    "Du_doan_ngay_thuong": y_pred_ngay,
    "Sai_so": abs(y_test_ngay.values - y_pred_ngay)
})

result_compare.head(10)

joblib.dump(model_ngay_thuong, "model_doanh_thu_ngay_thuong.pkl")
joblib.dump(model_cuoi_tuan, "model_doanh_thu_cuoi_tuan.pkl")
joblib.dump(model_tuan, "model_doanh_thu_tuan.pkl")

print("Đã lưu 3 model thành công!")

new_shop = pd.DataFrame([{
    "loai_quan": "cafe",
    "khu_vuc": "q10",
    "gan_truong": 1,
    "gan_van_phong": 1,
    "mat_tien": 1,
    "trong_hem": 0,
    "dien_tich_num": 100,
    "so_cho_ngoi_num": 50,
    "so_nhan_vien_num": 5,
    "rating_num": 4.5,
    "so_review_num": 300,
    "delivery": 1,
    "wifi": 1,
    "may_lanh": 1,
    "cho_ngoi_lau": 1,
    "tep_khach": "sinh_vien",
    "gio_peak": "14h-21h",
    "doi_thu_num": 6,
    "vi_tri_score": 2,
    "gia_tb_ngay_thuong": 35000,
    "gia_tb_cuoi_tuan": 42000
}])

du_doan_ngay_thuong = model_ngay_thuong.predict(new_shop)[0]
du_doan_cuoi_tuan = model_cuoi_tuan.predict(new_shop)[0]
du_doan_tuan = model_tuan.predict(new_shop)[0]

print("KẾT QUẢ DỰ ĐOÁN")
print("==============================")
print(f"Doanh thu ngày thường dự đoán: {du_doan_ngay_thuong:,.0f} VNĐ")
print(f"Doanh thu cuối tuần dự đoán  : {du_doan_cuoi_tuan:,.0f} VNĐ")
print(f"Doanh thu tuần dự đoán       : {du_doan_tuan:,.0f} VNĐ")

df_result = df.copy()

df_result["du_doan_doanh_thu_ngay_thuong"] = model_ngay_thuong.predict(X)
df_result["du_doan_doanh_thu_cuoi_tuan"] = model_cuoi_tuan.predict(X)
df_result["du_doan_doanh_thu_tuan"] = model_tuan.predict(X)

df_result["sai_so_ngay_thuong"] = abs(
    df_result["doanh_thu_ngay_thuong"] - df_result["du_doan_doanh_thu_ngay_thuong"]
)

df_result["sai_so_cuoi_tuan"] = abs(
    df_result["doanh_thu_cuoi_tuan"] - df_result["du_doan_doanh_thu_cuoi_tuan"]
)

df_result["sai_so_tuan"] = abs(
    df_result["doanh_thu_tuan"] - df_result["du_doan_doanh_thu_tuan"]
)

df_result.head()

output_file = "ket_qua_du_doan_doanh_thu.xlsx"

df_result.to_excel(output_file, index=False)

print("Đã xuất file:", output_file)
files.download(output_file)
files.download("model_doanh_thu_ngay_thuong.pkl")
files.download("model_doanh_thu_cuoi_tuan.pkl")
files.download("model_doanh_thu_tuan.pkl")

Số dòng, số cột: (399, 41)
Danh sách cột:
['ten_quan', 'loai_quan', 'khu_vuc', 'gan_truong', 'gan_van_phong', 'mat_tien', 'trong_hem', 'dien_tich_m2', 'so_cho_ngoi', 'so_nhan_vien', 'gia_tb', 'rating_google', 'so_review', 'delivery', 'wifi', 'may_lanh', 'cho_ngoi_lau', 'tep_khach', 'gio_peak', 'doi_thu_500m', 'revenue_level', 'dien_tich_num', 'so_cho_ngoi_num', 'so_nhan_vien_num', 'rating_num', 'so_review_num', 'doi_thu_num', 'vi_tri_score', 'gia_tb_ngay_thuong', 'gia_tb_cuoi_tuan', 'khach_co_dinh_ngay_thuong', 'khach_vang_lai_ngay_thuong', 'don_online_ngay_thuong', 'tong_khach_ngay_thuong', 'doanh_thu_ngay_thuong', 'khach_co_dinh_cuoi_tuan', 'khach_vang_lai_cuoi_tuan', 'don_online_cuoi_tuan', 'tong_khach_cuoi_tuan', 'doanh_thu_cuoi_tuan', 'doanh_thu_tuan']

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 41 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>